<a href="https://colab.research.google.com/github/kachionyenokwe/TempSound/blob/main/notebooks/02_audiogen_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Transformers, Accelerate, and Soundfile
!pip install -q transformers diffusers accelerate soundfile librosa

import os
import json
import torch
import soundfile as sf
from transformers import AutoProcessor, MusicgenForConditionalGeneration

print("✓ Transformers audio pipeline loaded successfully.")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

✓ Transformers audio pipeline loaded successfully.
GPU Available: True
Device Name: Tesla T4


In [4]:
!pip install -q sentencepiece

In [16]:
import sys
import torch
import torchaudio

# 1. Import audiocraft and its internal loaders module
import audiocraft
import audiocraft.models.loaders

# 2. Patch __version__ directly on both the parent module AND the internal loaders module namespace
audiocraft.__version__ = '1.4.0a2'
audiocraft.models.loaders.audiocraft.__version__ = '1.4.0a2'

from audiocraft.models import AudioGen

# 3. Load AudioGen model onto T4 GPU
print("[*] Loading facebook/audiogen-medium onto T4 GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AudioGen.get_pretrained('facebook/audiogen-medium')
model.set_generation_params(duration=10.0)  # Generate 10-second audio clips

print("\n[✓] AudioGen medium loaded successfully on GPU!")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

[*] Loading facebook/audiogen-medium onto T4 GPU...


compression_state_dict.bin: reconstructing file:   0%|          |  0.00B /  236MB            

compression_state_dict.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


state_dict.bin: reconstructing file:   0%|          |  0.00B / 3.68GB            

...

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.95GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


[✓] AudioGen medium loaded successfully on GPU!
Device Name: Tesla T4


In [17]:
import os
import json
import torch
from google.colab import files
from audiocraft.data.audio import audio_write

# Create output folder
os.makedirs("generated_audio", exist_ok=True)

# Upload manifest if not present in runtime environment
if not os.path.exists("gold_standard_50.json"):
    print("[*] Please upload 'gold_standard_50.json':")
    uploaded = files.upload()

with open("gold_standard_50.json", "r", encoding="utf-8") as f:
    prompts = json.load(f)

print(f"[*] Starting AudioGen batch generation for {len(prompts)} prompts...\n")

for idx, sample in enumerate(prompts, 1):
    caption = sample['caption']
    sample_id = sample.get('sample_id', idx)

    print(f"[{idx}/{len(prompts)}] Generating audio for: '{caption}'")

    # Generate audio tensor on T4 GPU
    with torch.no_grad():
        wav = model.generate([caption])  # Generates 10-second clip

    # Save as 16kHz WAV file
    output_filename = f"generated_audio/sample_{sample_id:02d}_audiogen"
    audio_write(
        output_filename,
        wav[0].cpu(),
        model.sample_rate,
        strategy="loudness",
        loudness_compressor=True
    )

print("\n[✓] AudioGen batch generation complete! All clips saved in 'generated_audio/'.")

# Zip generated audio files for easy download
!zip -q -r generated_audio_audiogen.zip generated_audio
print("[✓] Created 'generated_audio_audiogen.zip'. Downloading...")
files.download("generated_audio_audiogen.zip")

[*] Please upload 'gold_standard_50.json':


Saving gold_standard_50.json to gold_standard_50.json
[*] Starting AudioGen batch generation for 50 prompts...

[1/50] Generating audio for: 'a man speaks with slight accent as a crowd in the background makes noise then silences'
[2/50] Generating audio for: 'a vehicle accelerates and a man yells then a siren sounds'
[3/50] Generating audio for: 'a vehicle engine revs and accelerates whining then a man laughs'
[4/50] Generating audio for: 'a man is speaking and then a gun is fired and cocked'
[5/50] Generating audio for: 'many loud squeaks or whistles sounding together, a vehicle passing, then a vehicle horn accompanies a male voice'
[6/50] Generating audio for: 'a man speaks while a water tap runs then a beep occurs'
[7/50] Generating audio for: 'a motorcycle engine idles then turns off'
[8/50] Generating audio for: 'a woman speaks and then a kid yells loudly as soft voices speak softly in the background'
[9/50] Generating audio for: 'an engine idles as a man speaks then it accelerate

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>